In [1]:
import gc
import os
import sys

import numpy as np
import optuna
import pandas as pd
import torch
from sklearn.model_selection import KFold

current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, ".."))
sys.path.append(parent_dir)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
device.type

'cpu'

In [3]:
%load_ext autoreload
%autoreload 2
from drGAT import No_atten_drGAT
from drGAT.load_data import load_data
from drGAT.sampler import RandomSampler

In [4]:
drugAct, pos_num, null_mask, S_d, S_c, S_g, A_cg, A_dg = load_data()

load nci


100%|██████████| 25/25 [00:03<00:00,  6.99it/s]


Done!


In [5]:
PATH = "../nci_data/"

In [10]:
def objective(trial):
    params = {
        "n_drug": S_d.shape[0],
        "n_cell": S_c.shape[0],
        "n_gene": S_g.shape[0],
        "dropout1": trial.suggest_categorical("dropout1", [0.1, 0.2, 0.3, 0.4, 0.5]),
        "dropout2": trial.suggest_categorical("dropout2", [0.1, 0.2, 0.3, 0.4, 0.5]),
        "hidden1": trial.suggest_categorical(
            "hidden1",
            [256, 512, 1028],
        ),
        "hidden2": trial.suggest_categorical(
            "hidden2",
            [
                128,
                256,
                512,
            ],
        ),
        "hidden3": trial.suggest_categorical(
            "hidden3",
            [
                64,
                128,
                256,
            ],
        ),
        # "epochs": trial.suggest_categorical("epochs", [10, 50, 100, 200, 500]),
        "epochs": 2,
        "heads": trial.suggest_categorical("heads", [1, 2, 3, 4, 5]),
        "activation": trial.suggest_categorical(
            "activation", ["relu", "gelu", "swish"]
        ),
        "optimizer": trial.suggest_categorical("optimizer", ["Adam", "AdamW", "SGD"]),
        "lr": trial.suggest_float("lr", 1e-5, 1e-2, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
        "scheduler": trial.suggest_categorical(
            "scheduler", [None, "Cosine", "Step", "Plateau"]
        ),
        "gnn_layer": trial.suggest_categorical(
            "gnn_layer",
            ["GCN", "MPNN"],
        ),
    }

    # スケジューラ関連パラメータの条件付き追加
    if params["scheduler"] == "Cosine":
        params["T_max"] = trial.suggest_int("T_max", 20, 50)
    elif params["scheduler"] == "Step":
        params["scheduler_gamma"] = trial.suggest_float("gamma_step", 0.1, 0.95)
        params["step_size"] = trial.suggest_int("step_size", 10, 30)
    elif params["scheduler"] == "Plateau":
        params["patience"] = trial.suggest_int("patience_plateau", 3, 10)
        params["threshold"] = trial.suggest_float(
            "thresh_plateau", 1e-4, 1e-2, log=True
        )

    if params["hidden1"] < params["hidden2"] or params["hidden2"] < params["hidden3"]:
        raise optuna.TrialPruned("Invalid layer size configuration")

    if params["optimizer"] in ["Adam", "AdamW"]:
        params["amsgrad"] = trial.suggest_categorical("amsgrad", [True, False])

    if params["optimizer"] == "SGD":
        params["momentum"] = trial.suggest_float("momentum", 0.8, 0.95)
        params["nesterov"] = trial.suggest_categorical("nesterov", [True, False])

    # 隠れ層サイズとバッチサイズの関係を制約
    if (params["hidden1"] > 512) and (params["hidden2"] > 256):
        raise optuna.TrialPruned("Memory intensive configuration")

    try:
        k = 5
        kfold = KFold(n_splits=k, shuffle=True, random_state=42)

        res = pd.DataFrame()
        for train_index, test_index in kfold.split(np.arange(pos_num)):
            sampler = RandomSampler(
                drugAct,
                train_index,
                test_index,
                null_mask,
                S_d,
                S_c,
                S_g,
                A_cg,
                A_dg,
                PATH,
            )
            _, best_metrics, _ = No_atten_drGAT.train(
                sampler, params=params, device=device, verbose=False
            )

            res = pd.concat(
                [
                    res,
                    pd.DataFrame(best_metrics, index=["acc", "f1", "auroc", "aupr"]).T,
                ]
            )

        return [float(i) for i in res.mean()]

    except RuntimeError as e:
        if "CUDA out of memory" in str(e):
            print(f"Pruned trial {trial.number}: CUDA OOM")

            # メモリ解放処理
            with torch.cuda.device("cuda"):
                torch.cuda.empty_cache()
            gc.collect()

            # Pruning通知
            raise optuna.TrialPruned(f"OOM at trial {trial.number}")

        else:
            raise e

In [11]:
name = "NCI"
study = optuna.create_study(
    directions=["maximize"] * 4,
    sampler=optuna.samplers.TPESampler(),
    pruner=optuna.pruners.HyperbandPruner(),
    storage="sqlite:///{}_{}.sqlite3".format(name, "GCN_MPNN"),
    study_name=name,
    load_if_exists=True,
)
study.optimize(objective, n_trials=100)

[I 2025-03-27 19:15:12,309] Using an existing study with name 'NCI' instead of creating a new one.
[I 2025-03-27 19:15:12,374] Trial 14 pruned. Invalid layer size configuration
[I 2025-03-27 19:15:12,421] Trial 15 pruned. Invalid layer size configuration
[I 2025-03-27 19:15:12,466] Trial 16 pruned. Invalid layer size configuration
[I 2025-03-27 19:15:12,510] Trial 17 pruned. Invalid layer size configuration
/Users/inouey2/miniconda3/envs/torch/lib/python3.10/site-packages/torch/amp/grad_scaler.py:132: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(


Using device: cpu


100%|██████████| 2/2 [00:30<00:00, 15.14s/it]


Best model found at epoch 2


/Users/inouey2/miniconda3/envs/torch/lib/python3.10/site-packages/torch/amp/grad_scaler.py:132: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(


...

Best model found at epoch 2


/Users/inouey2/miniconda3/envs/torch/lib/python3.10/site-packages/torch/amp/grad_scaler.py:132: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(


Using device: cpu


 50%|█████     | 1/2 [00:18<00:18, 18.02s/it]
[W 2025-03-27 19:48:44,520] Trial 22 failed with parameters: {'dropout1': 0.5, 'dropout2': 0.3, 'hidden1': 512, 'hidden2': 256, 'hidden3': 64, 'heads': 3, 'activation': 'gelu', 'optimizer': 'AdamW', 'lr': 1.1308974930817926e-05, 'weight_decay': 9.747834060379388e-06, 'scheduler': 'Plateau', 'gnn_layer': 'GCN', 'patience_plateau': 7, 'thresh_plateau': 0.009383268742661332, 'amsgrad': False} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/Users/inouey2/miniconda3/envs/torch/lib/python3.10/site-packages/optuna/study/_optimize.py", line 197, in _run_trial
    value_or_values = func(trial)
  File "/var/folders/y3/ssnk1ytd3m5bjmrchh2lt74srg76p8/T/ipykernel_55986/2442631780.py", line 90, in objective
    _, best_metrics, _ = No_atten_drGAT.train(
  File "/Users/inouey2/code/drGAT/drGAT/No_atten_drGAT.py", line 188, in train
    train_one_epoch(
  File "/Users/inouey2/code/drGAT/drGAT/No_atten_drGAT.

KeyboardInterrupt: 

## Eval

In [ ]:
# test_drug = test_data.values[:, 0]
# test_cell = test_data.values[:, 1]

# test_labels = np.load("data/test_labels.npy")
# test_labels = torch.tensor(test_labels).float()
# test = [drug, cell, gene, edge_index, test_drug, test_cell, test_labels]

In [ ]:
# prob, res, test_attention = drGAT.eval(model, test)

In [21]:
?No_atten_drGAT.train

Signature:
No_atten_drGAT.train(
    sampler,
    params=None,
    is_sample=False,
    device=None,
    is_save=False,
    verbose=True,
)
Docstring: <no docstring>
File:      ~/code/drGAT/drGAT/No_atten_drGAT.py
Type:      function